# 📊 Member 2: Stratified Random Sampling (Proportional vs Neyman Allocation)
**Project:** Design, Estimation and Efficiency Comparison of Sampling Techniques on IPL Batting Data (2008–2024)

### Objectives:
1. Define 4 strata using batting positions: Openers (1-2), Middle (3-5), Lower-Middle (6-7), Tail (8-11).
2. Compute Proportional ($n_h \propto N_h$) and Neyman Optimal ($n_h \propto N_h S_h$) allocations.
3. Draw stratified samples and calculate combined estimators $\bar{y}_{st}$ and $V(\bar{y}_{st})$.
4. Evaluate precision gains over Simple Random Sampling.
5. Conduct One-Way ANOVA across strata to test for differences in stratum means.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from sampling.frame import load_frame
from sampling import stratified, estimators as est
from sampling.tests import anova_strata

frame = load_frame()
print(f"Loaded frame with {len(frame):,} rows.")


## 1. Strata Characteristics and Allocations
Under Neyman allocation:
$$n_h = n \cdot \frac{N_h S_h}{\sum N_h S_h}$$


In [ ]:
alloc_prop = stratified.compute_allocation(frame, config.n, "proportional")
alloc_prop.attrs["method"] = "proportional"

alloc_neyman = stratified.compute_allocation(frame, config.n, "neyman")
alloc_neyman.attrs["method"] = "neyman"

comparison_alloc = pd.merge(
    alloc_prop[["stratum", "Nh", "Wh", "Sh", "nh"]].rename(columns={"nh": "nh_proportional"}),
    alloc_neyman[["stratum", "nh"]].rename(columns={"nh": "nh_neyman"}),
    on="stratum"
)
comparison_alloc


## 2. Drawing Stratified Samples & Estimation
Combined Stratified Mean and Variance:
$$\bar{y}_{st} = \sum_{h=1}^L W_h \bar{y}_h, \quad V(\bar{y}_{st}) = \sum_{h=1}^L W_h^2 \left( 1 - \frac{n_h}{N_h} \right) \frac{s_h^2}{n_h}$$


In [ ]:
true_mean = float(frame["runs"].mean())
true_total = float(frame["runs"].sum())
true_prop = float((frame["runs"] >= 30).mean())

# Proportional
s_sp = stratified.draw_stratified(frame, alloc_prop, rng=np.random.default_rng(config.SEED + 2))
res_prop = stratified.analyse_stratified(s_sp, frame, alloc_prop, config.N, true_mean, true_total, true_prop)

# Neyman
s_sn = stratified.draw_stratified(frame, alloc_neyman, rng=np.random.default_rng(config.SEED + 3))
res_neyman = stratified.analyse_stratified(s_sn, frame, alloc_neyman, config.N, true_mean, true_total, true_prop)

# Reference SRS variance
var_srs = (1 - config.n / config.N) * float(frame["runs"].var(ddof=1)) / config.n

print("=== Proportional Allocation ===")
print(f"Mean: {res_prop['mean']:.4f}, SE: {res_prop['se_mean']:.4f}, 95% CI: {res_prop['ci_mean']}")
print(f"RE vs SRS: {var_srs / res_prop['var_mean']:.4f}")

print("\n=== Neyman Allocation ===")
print(f"Mean: {res_neyman['mean']:.4f}, SE: {res_neyman['se_mean']:.4f}, 95% CI: {res_neyman['ci_mean']}")
print(f"RE vs SRS: {var_srs / res_neyman['var_mean']:.4f}")


## 3. One-Way ANOVA across Batting Position Strata


In [ ]:
anova_res = anova_strata(frame)
print(f"Test: {anova_res['test']}")
print(f"F-value: {anova_res['F_value']:.4f}, p-value: {anova_res['p_value']:.4e}")
print(f"Eta-squared (η²): {anova_res['eta_squared']:.4f}")
print(f"Decision: {anova_res['decision']}")


## 4. Stratified Allocation Visualisation


In [ ]:
x = np.arange(len(comparison_alloc))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, comparison_alloc["nh_proportional"], width, label="Proportional", color="#2A9D8F")
ax.bar(x + width/2, comparison_alloc["nh_neyman"], width, label="Neyman", color="#E76F51")
ax.set_xticks(x)
ax.set_xticklabels(comparison_alloc["stratum"])
ax.set_ylabel("Sample Size (n_h)")
ax.set_title("Stratified Allocation: Proportional vs Neyman")
ax.legend()
plt.tight_layout()
plt.show()
